In [2]:
!hostname

n38.clstr


In [4]:
import pandas as pd 
import requests
import io
import numpy as np
import matplotlib.pyplot as plt
import glob
import xarray as xr
import geopandas as gpd
import matplotlib.colors as mcolors
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [1]:
winter_months = [11, 12, 1, 2, 3]

Alaska Shapefiles 
-

In [5]:
shapefile_path = "/center1/DYNDOWN/phutton5/ROS/boundaries/Alaska_Borough_and_Census_Area_Boundaries.shp"
borough_boundaries = gpd.read_file(shapefile_path)
borough_boundaries = borough_boundaries.set_crs(epsg=3338)
borough_boundaries = borough_boundaries.to_crs(epsg=4326)
FNSB_boundary = borough_boundaries[borough_boundaries['CommunityN'] == 'Fairbanks North Star Borough']
FNSB_geom = FNSB_boundary.geometry.iloc[0] 
FNSB_coords = []
FNSB_coords.extend(list(FNSB_geom.exterior.coords))
FNSB_coords = np.array(FNSB_coords)  
FNSB_coords = pd.DataFrame({
    "lon": FNSB_coords[:, 0],
    "lat": FNSB_coords[:, 1]})

Fairbanks_lat=(64.84)
Fairbanks_lon=(-147.72)

ASOS 
-

In [70]:

#Bethel
#website='https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?network=AK_ASOS&station=PABE&data=tmpc&data=p01m&data=wxcodes&data=snowdepth&year1=1949&month1=11&day1=1&year2=2023&month2=3&day2=31&tz=America%2FAnchorage&format=onlycomma&latlon=yes&elev=no&missing=M&trace=0.0001&direct=no&report_type=3&report_type=4'
#Fairbanks
#website='https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?network=AK_ASOS&station=PAFA&data=p01m&data=wxcodes&data=metar&year1=1950&month1=1&day1=1&year2=2023&month2=3&day2=31&tz=Etc%2FUTC&format=onlycomma&latlon=yes&elev=yes&missing=M&trace=T&direct=no&report_type=3&report_type=4'
#Anc
#website='https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?network=AK_ASOS&station=PANC&data=tmpc&data=p01m&data=wxcodes&data=snowdepth&year1=1949&month1=10&day1=1&year2=2023&month2=1&day2=1&tz=America%2FAnchorage&format=onlycomma&latlon=yes&elev=no&missing=M&trace=0.0001&direct=no&report_type=3&report_type=4'
#Talk
website='https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?network=AK_ASOS&station=PATK&data=p01m&data=wxcodes&data=metar&year1=1950&month1=10&day1=1&year2=2023&month2=3&day2=31&tz=Etc%2FUTC&format=onlycomma&latlon=yes&elev=no&missing=M&trace=T&direct=no&report_type=3&report_type=4'

response = requests.get(website)
if response.status_code == 200:
    data = io.StringIO(response.text)
    df = pd.read_csv(data, comment="#")  
    print(df.head())
else:
    print("Error:", response.status_code)

  station             valid      lon    lat p01m wxcodes  \
0    PATK  1950-10-01 00:00 -150.095  62.32    M       M   
1    PATK  1950-10-01 01:00 -150.095  62.32    M       M   
2    PATK  1950-10-01 02:00 -150.095  62.32    M       M   
3    PATK  1950-10-01 03:00 -150.095  62.32    M       M   
4    PATK  1950-10-01 04:00 -150.095  62.32    M       M   

                                               metar  
0  PATK 010000Z AUTO 60SM 11/06 RMK SLP163 T01110...  
1  PATK 010100Z AUTO 60SM 10/06 RMK SLP156 T01000...  
2  PATK 010200Z AUTO 40SM 10/06 RMK SLP152 T01000...  
3  PATK 010300Z AUTO 30SM 08/06 RMK SLP149 T00830...  
4  PATK 010400Z AUTO 15SM 07/05 RMK SLP146 T00720...  


In [77]:
df['valid'] = pd.to_datetime(df['valid'])
df['month'] = df['valid'].dt.month
df['date'] = df['valid'].dt.date
df['time'] = df['valid'].dt.time
#df['tmpc']
#df = df.drop(columns=['column_name'])
winter_df = df[df['month'].isin([11, 12, 1, 2, 3])] #filter to only keep the ROS months 

winter_df['date'] = pd.to_datetime(winter_df['date'])
year = winter_df['date'].dt.year
month = winter_df['date'].dt.month
season_start = year.where(~month.isin([1, 2, 3]), year - 1)
season_end = season_start + 1
winter_df['season'] = season_start.astype(str) + '-' + season_end.astype(str)

#filter to only when RA is present 
mask = winter_df['wxcodes'].str.contains('RA', na=False)
rain_and_mixed_df = winter_df[mask]
rain_and_mixed_df['p01m'] = rain_and_mixed_df['p01m'].replace('M', np.nan) 
rain_and_mixed_df['p01m'] = rain_and_mixed_df['p01m'].replace('T', 0.001) 
rain_and_mixed_df['p01m'] = pd.to_numeric(rain_and_mixed_df['p01m'], errors='coerce')

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_238021/1912088275.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  winter_df['date'] = pd.to_datetime(winter_df['date'])
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_238021/1912088275.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  winter_df['season'] = season_start.astype(str) + '-' + season_end.astype(str)
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_238021/1912088275.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [78]:
monthly_order = [11, 12, 1, 2, 3]

seasonal_sum_rain_and_mixed_df = (rain_and_mixed_df.groupby('season')['p01m'].sum())
monthly_sum_by_season_ASOS = (rain_and_mixed_df.groupby(['month'])['p01m'].sum()) 
monthly_sum_by_season_ASOS=monthly_sum_by_season_ASOS.loc[monthly_order]

monthly_mean_by_season_ASOS = rain_and_mixed_df.groupby(['season'])['p01m'].mean()

start_years = seasonal_sum_rain_and_mixed_df.index.str.slice(0, 4).astype(int)
all_seasons = [f"{y}-{y+1}" for y in range(start_years.min(), start_years.max() + 1)]

seasonal_sum_rain_and_mixed_df = seasonal_sum_rain_and_mixed_df.reindex(all_seasons)
seasonal_sum_rain_and_mixed_df=seasonal_sum_rain_and_mixed_df.fillna(0)
seasons_seasonal_sum_rain_and_mixed_df = seasonal_sum_rain_and_mixed_df.index.tolist()

In [79]:
seasonal_sum_rain_and_mixed_df

season
1950-1951     0.000
1951-1952     0.000
1952-1953     0.000
1953-1954     0.000
1954-1955     0.000
              ...  
2018-2019    36.513
2019-2020    44.599
2020-2021     9.896
2021-2022    50.721
2022-2023     9.371
Name: p01m, Length: 73, dtype: float64

In [80]:
no_zero_seasonal_sum_rain_and_mixed_df = seasonal_sum_rain_and_mixed_df[
    seasonal_sum_rain_and_mixed_df != 0]

In [81]:
np.mean(no_zero_seasonal_sum_rain_and_mixed_df)

47.46355555555555

In [82]:
std = no_zero_seasonal_sum_rain_and_mixed_df.std(skipna=True, ddof=1)
print('std', std)
se = std / np.sqrt(no_zero_seasonal_sum_rain_and_mixed_df.count())
print('se', se)

std 54.85425210103682
se 10.556705738909734


In [83]:
print(np.mean(seasonal_sum_rain_and_mixed_df))
std = seasonal_sum_rain_and_mixed_df.std(skipna=True, ddof=1)
print('std', std)
se = std / np.sqrt(seasonal_sum_rain_and_mixed_df.count())
print('se', se)

17.555013698630137
std 40.23576149794178
se 4.709239684029007


In [8]:
seasonal_sum_rain_and_mixed_df = (rain_and_mixed_df.groupby('season')['p01m'].sum())
monthly_sum_ASOS = (rain_and_mixed_df.groupby(['month'])['p01m'].sum())
monthly_sum_ASOS=monthly_sum_ASOS.reindex(winter_months).values 

monthly_mean_by_season_ASOS = (rain_and_mixed_df.groupby(['season', 'month'])['p01m'].mean().unstack('month'))

start_years = seasonal_sum_rain_and_mixed_df.index.str.slice(0, 4).astype(int)
all_seasons = [f"{y}-{y+1}" for y in range(start_years.min(), start_years.max() + 1)]

seasonal_sum_rain_and_mixed_df = seasonal_sum_rain_and_mixed_df.reindex(all_seasons)
#seasonal_sum_rain_and_mixed_df=seasonal_sum_rain_and_mixed_df.fillna(0)
seasons_seasonal_sum_rain_and_mixed_df = seasonal_sum_rain_and_mixed_df.index.tolist()

In [8]:
def getXY(lat, lon, dataarray):
    abslat = np.abs(dataarray.XLAT-lat)
    abslon = np.abs(dataarray.XLONG-lon)
    d = abslon**2 + abslat**2
    flat_index = np.argmin(d.values)
    yloc, xloc = np.unravel_index(flat_index, d.shape)
    return xloc, yloc

def getXY_latlon(lat, lon, dataarray):
    abslat = np.abs(dataarray.latitude - lat)
    abslon = np.abs(dataarray.longitude - lon)
    d = abslon**2 + abslat**2
    flat_index = np.argmin(d.values)
    yloc, xloc = np.unravel_index(flat_index, d.shape)
    return xloc, yloc

In [16]:
regridded_era5_path='/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/ERA5_31kmto4km_nearest_regridded.nc'
regridded_era5=xr.open_dataset(regridded_era5_path)

#ERA5  4km
era5_4km='/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/ROS_Monthly_*.nc'
era5_4km = xr.open_mfdataset(era5_4km,combine="by_coords", parallel=True)

In [17]:
x_idx, y_idx = getXY(Fairbanks_lat, Fairbanks_lon, regridded_era5)
nearest_lat = regridded_era5.XLAT[y_idx, x_idx]
nearest_lon = regridded_era5.XLONG[y_idx, x_idx]


In [18]:
cell = era5_4km.isel(
    south_north=y_idx,
    west_east=x_idx
)

In [19]:
era5_4km_rain_sum_at_site = era5_4km['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx)
seasonal_rain_sum_at_site_era5_4km=era5_4km_rain_sum_at_site.sum(dim='month')
monthly_rain_sum_at_site_era5_4km=era5_4km_rain_sum_at_site.sum(dim='season')
#monthly_rain_MEAN_at_site_era5_4km=era5_4km['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx).groupby('month','season').mean()

era5_regridded_31km_rain_sum_at_site = regridded_era5['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx)
seasonal_rain_sum_at_site_era5_regridded_31km=era5_regridded_31km_rain_sum_at_site.sum(dim='month')
monthly_rain_sum_at_site_era5_regridded_31km=era5_regridded_31km_rain_sum_at_site.sum(dim='season')

#monthly_rain_MEAN_at_site_era5_regridded_31km=regridded_era5['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx).groupby('month').mean()

seasons=era5_4km['season']

In [20]:
era5_4km_df = era5_4km_rain_sum_at_site.to_dataframe().reset_index()
era5_31km_df = era5_regridded_31km_rain_sum_at_site.to_dataframe().reset_index() 
era5_4km_monthly = era5_4km_df.groupby(['season', 'month'])['rain_ros_sum'].sum().reset_index()
era5_31km_monthly = era5_31km_df.groupby(['season', 'month'])['rain_ros_sum'].sum().reset_index()

top_rain_df = rick_clean.copy()

top_rain_df = top_rain_df.merge(
    filtered_rain_and_mixed_df,
    on=['season', 'month'],
    how='left').rename(columns={'p01m':'ASOS'})

top_rain_df = top_rain_df.merge(
    era5_4km_monthly,
    on=['season', 'month'],
    how='left').rename(columns={'rain_ros_sum':'Month_ERA5_4km_mm'})

top_rain_df = top_rain_df.merge(
    era5_31km_monthly,
    on=['season', 'month'],
    how='left').rename(columns={'rain_ros_sum':'Month_ERA5_31km_mm'})

top_rain_df = top_rain_df.groupby(['season', 'month'], as_index=False).sum()
top_rain_df.drop(top_rain_df[top_rain_df['season'] == '2024-2025'].index, inplace=True)
top_rain_df

,season,month,Local,ASOS,Month_ERA5_4km_mm,Month_ERA5_31km_mm
0,1954-1955,3,0.254,0.000,0.388934,0.000000
1,1956-1957,1,0.762,0.000,7.004359,0.392925
2,1960-1961,1,1.016,0.000,0.000000,0.000000
3,1962-1963,1,14.478,12.410,18.590502,15.130484
4,1962-1963,12,7.112,2.290,4.077709,0.418650
5,1966-1967,3,6.604,0.250,5.808046,0.926677
6,1967-1968,11,2.540,3.040,7.817150,1.143859
7,1967-1968,12,10.922,4.300,21.801321,8.359328
8,1969-1970,2,1.270,0.510,1.397190,0.287160
9,1970-1971,12,5.588,4.060,19.924116,0.000000


In [22]:
#ERA5  4km
dec_2021_era5_4km='/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/ROS_Dec2021_hourly_4km.nc' 
dec_2021_era5_4km = xr.open_mfdataset(dec_2021_era5_4km,combine="by_coords", parallel=True)

dec_2021_era5_4km = dec_2021_era5_4km.sel(Time=slice("2021-12-26", "2021-12-27 23:59:59"))

lat=dec_2021_era5_4km['XLAT'].values
lon=dec_2021_era5_4km['XLONG'].values

x_idx, y_idx = getXY(Fairbanks_lat, Fairbanks_lon, dec_2021_era5_4km)
nearest_lat = dec_2021_era5_4km.XLAT[y_idx, x_idx]
nearest_lon = dec_2021_era5_4km.XLONG[y_idx, x_idx]

dec2021_cell4km = dec_2021_era5_4km.isel(
    south_north=y_idx,
    west_east=x_idx)

dec2021_cell4km['RAIN'].sum().values.item()

22.734561920166016

In [33]:
import pandas as pd

def convert_utc_to_ak(da):
    # Convert DataArray to pandas Series, localize to UTC, convert to AK time
    series = da.to_series()
    series_utc = pd.to_datetime(series, utc=True)
    series_ak = series_utc.dt.tz_convert('America/Anchorage')
    return series_ak.values  # or keep as Series if preferred

cell['Time_ak'] = convert_utc_to_ak(cell['Time'])


In [34]:
time_utc = pd.to_datetime(cell['Time'].values, utc=True)
time_ak = time_utc.tz_convert('America/Anchorage')

print(time_ak[:3])

DatetimeIndex(['1955-02-28 14:00:00-10:00', '1955-02-28 15:00:00-10:00',
               '1955-02-28 16:00:00-10:00'],
              dtype='datetime64[ns, America/Anchorage]', freq=None)


In [35]:
time_ak_naive = time_ak.tz_localize(None)  # drops tz info but keeps the shifted values
cell['Time_ak'] = ('time', time_ak_naive)
cell['Time_ak']

<xarray.DataArray 'Time_ak' (time: 27528)> Size: 220kB
array(['1955-02-28T14:00:00.000000000', '1955-02-28T15:00:00.000000000',
       '1955-02-28T16:00:00.000000000', ..., '2022-11-30T12:00:00.000000000',
       '2022-11-30T13:00:00.000000000', '2022-11-30T14:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
    XLAT     float32 4B dask.array<chunksize=(), meta=np.ndarray>
    XLONG    float32 4B dask.array<chunksize=(), meta=np.ndarray>
    Time_ak  (time) datetime64[ns] 220kB 1955-02-28T14:00:00 ... 2022-11-30T1...
Dimensions without coordinates: time

cell_ak = cell.assign_coords(Time_ak=('Time', cell['Time_ak'].values))
cell_ak = cell_ak.swap_dims({'Time': 'Time_ak'})